# 数据读取 （data from Unit03_1_5_select.mat）

## 原始数据的读取

In [1]:
import scipy.io
import numpy as np

# ==================== 1️⃣ 读取 .mat 文件 ====================
mat_data = scipy.io.loadmat(
    '/home/charles/HZU/Data_processed/multi-condition-transfer-learning/Unit03_1_5_select_4.mat'
)

data = mat_data['Unit03_1_5_select_4']

print("Original data shape:", data.shape)  # (N, 14)

# ==================== 2️⃣ 设置抽样参数 ====================
n = 10000        # ⭐你只需要改这里
seed = 42      # 可选：保证可复现

np.random.seed(seed)

num_samples = data.shape[0]
assert n <= num_samples, "n cannot be larger than total samples!"

# ==================== 3️⃣ 随机抽取 n 个样本（行） ====================
indices = np.random.choice(num_samples, size=n, replace=False)
data = data[indices, :]

print("Sampled data shape:", data.shape)

# ==================== 4️⃣ 查看抽样结果 ====================
print(data[:5])   # 前 5 行看看


Original data shape: (120000, 14)
Sampled data shape: (10000, 14)
[[ 6.44879007e+00  9.89900780e+00  1.90707862e-01 -1.08997412e+01
   2.62183411e+02  3.11502747e+02  5.61110718e+02  8.06992188e+02
   7.92810608e+02  8.95006958e+02  7.50036987e+02  7.48097046e+02
   7.48671387e+02  2.98820686e+00]
 [ 6.53146410e+00  9.83751106e+00  1.89284369e-01 -1.14709768e+01
   2.62158813e+02  3.12536133e+02  5.60454041e+02  8.14469238e+02
   8.00921570e+02  8.94770813e+02  7.51456848e+02  7.48712585e+02
   7.49641296e+02  4.64089346e+00]
 [ 6.56376362e+00  9.10135174e+00  1.71239838e-01 -1.40106335e+01
   2.62777527e+02  3.08556671e+02  5.62043396e+02  8.09128113e+02
   7.98723511e+02  8.97193604e+02  7.49260986e+02  7.47456848e+02
   7.48115967e+02  2.35006785e+00]
 [ 6.33066225e+00  9.98306084e+00  1.95128351e-01 -1.39515629e+01
   2.63632294e+02  3.07687988e+02  5.75290466e+02  8.21349548e+02
   8.02239624e+02  9.05823181e+02  7.48023193e+02  7.48449585e+02
   7.47242065e+02  8.73057747e+00]
 [

## 特征和标签的分离

In [2]:
import numpy as np

# 假设 'data' 是一个二维数组或矩阵
# 分离特征和标签

# 特征是除了最后一列的数据
X = data[:, :-1]  # 所有行，去除最后一列

# 标签是最后一列的数据
y = data[:, -1]  # 所有行，只取最后一列
y = y.reshape(-1, 1)

# # 查看特征和标签
# print("Features (X):")
# print(X[:5])  # 查看前5个特征样本
# print("Labels (y):")
# print(y[:5])  # 查看前5个标签

# 查看特征和标签的形状
print("Shape of Features (X):", X.shape)
print("Shape of Labels (y):", y.shape)

Shape of Features (X): (10000, 13)
Shape of Labels (y): (10000, 1)


## 三集划分

In [3]:
import numpy as np
from sklearn.model_selection import train_test_split

# 假设 X 和 y 是已经分离好的特征和标签
# X: 特征数据，y: 标签数据

# 设置随机种子，确保结果可复现
random_seed = 42

# 控制三集的划分比例：例如 70% 训练集，15% 验证集，15% 测试集
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

# 确保划分比例之和为1
assert train_ratio + val_ratio + test_ratio == 1.0, "The sum of ratios must be 1."

# 第一次划分，将训练集和验证+测试集合并
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=1 - train_ratio, random_state=random_seed)

# 第二次划分，将验证集和测试集分开
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=test_ratio / (val_ratio + test_ratio), random_state=random_seed)

# 打印各个数据集的形状
print("Shape of Training Set (X_train, y_train):", X_train.shape, y_train.shape)
print("Shape of Validation Set (X_val, y_val):", X_val.shape, y_val.shape)
print("Shape of Test Set (X_test, y_test):", X_test.shape, y_test.shape)


Shape of Training Set (X_train, y_train): (6999, 13) (6999, 1)
Shape of Validation Set (X_val, y_val): (1500, 13) (1500, 1)
Shape of Test Set (X_test, y_test): (1501, 13) (1501, 1)


## 归一化

In [4]:
import numpy as np

# ======================= 1️⃣ X：用 train 的均值和方差 =======================
mu = X_train.mean(axis=0, keepdims=True)     # (1, 13)
std = X_train.std(axis=0, keepdims=True)    # (1, 13)
std[std == 0] = 1e-8

X_train = (X_train - mu) / std
X_val   = (X_val   - mu) / std
X_test  = (X_test  - mu) / std

print("X normalized shapes:")
print(X_train.shape, X_val.shape, X_test.shape)


# ======================= 2️⃣ y：同样只用 train =======================
y_mu = y_train.mean(axis=0, keepdims=True)     # (1,1)
y_std = y_train.std(axis=0, keepdims=True)    # (1,1)
y_std[y_std == 0] = 1e-8

#===========================保留归一化前的y值==============================#
y_train_raw = y_train
y_val_raw = y_val
y_test_raw = y_test


y_train = (y_train - y_mu) / y_std
y_val   = (y_val   - y_mu) / y_std
y_test  = (y_test  - y_mu) / y_std

print("y normalized shapes:")
print(y_train.shape, y_val.shape, y_test.shape)


# ======================= 3️⃣ 反归一化函数 =======================
def inverse_y(y_norm, y_mu, y_std):
    """
    y_norm: normalized prediction, shape [N,1] or [N]
    y_mu, y_std: from training set
    """
    return y_norm * y_std + y_mu


X normalized shapes:
(6999, 13) (1500, 13) (1501, 13)
y normalized shapes:
(6999, 1) (1500, 1) (1501, 1)


## 样本滑窗

In [5]:
import torch
import numpy as np

def sliding_window(
    X,
    y,
    window_size,
    stride=1
):
    """
    X: [N, D]
    y: [N] or [N, 1]
    window_size: T
    stride: step between windows
    """

    if isinstance(X, np.ndarray):
        X = torch.from_numpy(X).float()
    if isinstance(y, np.ndarray):
        y = torch.from_numpy(y).float()

    assert X.dim() == 2
    assert len(X) == len(y)

    X_seq, y_seq = [], []

    for end in range(window_size - 1, len(X), stride):
        start = end - window_size + 1
        X_seq.append(X[start:end + 1])
        y_seq.append(y[end])

    return (
        torch.stack(X_seq),           # [N', T, D]
        torch.stack(y_seq).view(-1, 1)
    )


T = 20  # 滑窗长度，你之后可以调

X_train = torch.from_numpy(X_train).float()
y_train = torch.from_numpy(y_train).float()#每一个滑窗样本的标签 y，取的是“窗口最后一个时间点”的真实值

X_val   = torch.from_numpy(X_val).float()
y_val   = torch.from_numpy(y_val).float()

X_test  = torch.from_numpy(X_test).float()
y_test  = torch.from_numpy(y_test).float()


X_train_seq, y_train_seq = sliding_window(
    X_train, y_train, window_size=T
)
X_val_seq, y_val_seq = sliding_window(
    X_val, y_val, window_size=T
)
X_test_seq, y_test_seq = sliding_window(
    X_test, y_test, window_size=T
)
print(X_train_seq.shape)  # [N', T, D]
print(y_train_seq.shape)  # [N', 1]


torch.Size([6980, 20, 13])
torch.Size([6980, 1])


# 模型

## RNN模型定义

In [6]:
import torch
import torch.nn as nn

class KANLinear(nn.Module):
    """
    KAN-style linear layer:
    y = sum_i f_i(x_i), where f_i is a learnable 1D function
    implemented via basis expansion.
    """
    def __init__(self, in_features, num_basis=8):
        super().__init__()
        self.in_features = in_features
        self.num_basis = num_basis

        # 每个维度一组 basis 权重
        self.weight = nn.Parameter(
            torch.randn(in_features, num_basis) * 0.1
        )
        self.bias = nn.Parameter(torch.zeros(1))

        # 固定 basis centers（[-1, 1]）
        centers = torch.linspace(-1, 1, num_basis)
        self.register_buffer("centers", centers)

        self.gamma = nn.Parameter(torch.ones(1))  # 控制平滑度

    def forward(self, x):
        """
        x: [B, in_features]
        """
        # x -> [B, in_features, num_basis]
        x_exp = x.unsqueeze(-1)

        # RBF basis
        basis = torch.exp(
            -self.gamma * (x_exp - self.centers) ** 2
        )

        # 加权求和
        y = (basis * self.weight).sum(dim=(1, 2)) + self.bias
        return y.unsqueeze(-1)


class RNNRegressor(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim=64,
        num_layers=1,
        rnn_type="LSTM",
        dropout=0.0,
        kan_basis=8        # ⭐ KAN 的基函数数量
    ):
        super().__init__()

        self.rnn_type = rnn_type.upper()

        if self.rnn_type == "RNN":
            self.rnn = nn.RNN(
                input_dim, hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                dropout=dropout if num_layers > 1 else 0.0
            )
        elif self.rnn_type == "LSTM":
            self.rnn = nn.LSTM(
                input_dim, hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                dropout=dropout if num_layers > 1 else 0.0
            )
        elif self.rnn_type == "GRU":
            self.rnn = nn.GRU(
                input_dim, hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                dropout=dropout if num_layers > 1 else 0.0
            )
        else:
            raise ValueError("rnn_type must be RNN / LSTM / GRU")

        # ================= 回归头：KAN =================
        self.regressor = KANLinear(
            in_features=hidden_dim,
            num_basis=kan_basis
        )

    def forward(self, x):
        """
        x: [B, T, D]
        """
        out, _ = self.rnn(x)
        h_last = out[:, -1, :]     # [B, H]
        y_hat = self.regressor(h_last)
        return y_hat


    
def train_one_epoch(model, optimizer, criterion, X, y, batch_size=64):
    model.train()
    total_loss = 0.0

    for i in range(0, len(X), batch_size):
        xb = X[i:i+batch_size]
        yb = y[i:i+batch_size]

        optimizer.zero_grad()
        y_hat = model(xb).view(-1)
        loss = criterion(y_hat, yb.view(-1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)

    return total_loss / len(X)


@torch.no_grad()
def evaluate(model, criterion, X, y, batch_size=64):
    model.eval()
    total_loss = 0.0

    for i in range(0, len(X), batch_size):
        xb = X[i:i+batch_size]
        yb = y[i:i+batch_size]

        y_hat = model(xb).view(-1)
        loss = criterion(y_hat, yb.view(-1))

        total_loss += loss.item() * xb.size(0)

    return total_loss / len(X)

from sklearn.metrics import r2_score

@torch.no_grad()
def evaluate_r2(model, X, y, batch_size=64):
    model.eval()

    y_true_list = []
    y_pred_list = []

    for i in range(0, len(X), batch_size):
        xb = X[i:i+batch_size]
        yb = y[i:i+batch_size]

        y_hat = model(xb)

        y_true_list.append(yb.view(-1).cpu().numpy())
        y_pred_list.append(y_hat.view(-1).cpu().numpy())

    y_true = np.concatenate(y_true_list)
    y_pred = np.concatenate(y_pred_list)

    return r2_score(y_true, y_pred)


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 假设你已经是 torch.Tensor
X_train_seq = X_train_seq.to(device)
y_train_seq = y_train_seq.to(device)
X_val_seq   = X_val_seq.to(device)
y_val_seq   = y_val_seq.to(device)



model = RNNRegressor(
    input_dim=X_train_seq.shape[-1],
    hidden_dim=64,
    num_layers=2,
    rnn_type="LSTM",
    dropout=0.1,
    kan_basis=8
)

model = model.to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 50

best_val_r2 = -float("inf")   # R² 越大越好
best_epoch = -1

for epoch in range(1, num_epochs + 1):

    # ===== 1️⃣ 训练 =====
    train_loss = train_one_epoch(
        model, optimizer, criterion,
        X_train_seq, y_train_seq
    )

    # ===== 2️⃣ 验证 MSE =====
    val_loss = evaluate(
        model, criterion,
        X_val_seq, y_val_seq
    )

    # ===== 3️⃣ 计算 R² =====
    train_r2 = evaluate_r2(
        model, X_train_seq, y_train_seq
    )
    val_r2 = evaluate_r2(
        model, X_val_seq, y_val_seq
    )

    # ===== 4️⃣ 保存最优模型（按 val R²）=====
    if val_r2 > best_val_r2:
        best_val_r2 = val_r2
        best_epoch = epoch
        torch.save(model.state_dict(), "/home/charles/HZU/Industrial_Software_Testing/Industrial_Software_Testing/multi_condition_transfer_learning/Single_condition_Regression_v2/result/model_save/best_RNN_model.pt")

    # ===== 5️⃣ 打印日志 =====
    print(
        f"[Epoch {epoch:03d}] "
        f"Train MSE: {train_loss:.4f} | "
        f"Val MSE: {val_loss:.4f} | "
        f"Train R²: {train_r2:.4f} | "
        f"Val R²: {val_r2:.4f}"
    )

print(
    f"\n✅ Best model saved at epoch {best_epoch}, "
    f"Val R² = {best_val_r2:.4f}"
)



[Epoch 001] Train MSE: 0.8868 | Val MSE: 0.6544 | Train R²: 0.3391 | Val R²: 0.3293
[Epoch 002] Train MSE: 0.6603 | Val MSE: 0.6302 | Train R²: 0.3687 | Val R²: 0.3542
[Epoch 003] Train MSE: 0.6399 | Val MSE: 0.6168 | Train R²: 0.3856 | Val R²: 0.3678
[Epoch 004] Train MSE: 0.6254 | Val MSE: 0.6064 | Train R²: 0.3995 | Val R²: 0.3785
[Epoch 005] Train MSE: 0.6120 | Val MSE: 0.5975 | Train R²: 0.4116 | Val R²: 0.3876
[Epoch 006] Train MSE: 0.6023 | Val MSE: 0.5886 | Train R²: 0.4252 | Val R²: 0.3968
[Epoch 007] Train MSE: 0.5883 | Val MSE: 0.5841 | Train R²: 0.4332 | Val R²: 0.4013
[Epoch 008] Train MSE: 0.5773 | Val MSE: 0.5793 | Train R²: 0.4433 | Val R²: 0.4063
[Epoch 009] Train MSE: 0.5722 | Val MSE: 0.5777 | Train R²: 0.4478 | Val R²: 0.4079
[Epoch 010] Train MSE: 0.5629 | Val MSE: 0.5726 | Train R²: 0.4596 | Val R²: 0.4132
[Epoch 011] Train MSE: 0.5599 | Val MSE: 0.5695 | Train R²: 0.4637 | Val R²: 0.4163
[Epoch 012] Train MSE: 0.5482 | Val MSE: 0.5669 | Train R²: 0.4719 | Val R²:

# 测试

In [9]:
# ======================= 单 cell：Test 评估（device-safe） =======================
import torch
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error

# ---------- 1️⃣ 统一 device ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)
X_test_seq = X_test_seq.to(device)
y_test_seq = y_test_seq.to(device)

# ---------- 2️⃣ 加载最优模型 ----------
model.load_state_dict(torch.load("/home/charles/HZU/Industrial_Software_Testing/Industrial_Software_Testing/multi_condition_transfer_learning/Single_condition_Regression_v2/result/model_save/best_RNN_model.pt", map_location=device))
model.eval()

# ---------- 3️⃣ Test 预测 ----------
y_true_list = []
y_pred_list = []

batch_size = 64

with torch.no_grad():
    for i in range(0, len(X_test_seq), batch_size):
        xb = X_test_seq[i:i+batch_size]
        yb = y_test_seq[i:i+batch_size]

        y_hat = model(xb)

        y_true_list.append(yb.view(-1).cpu().numpy())
        y_pred_list.append(y_hat.view(-1).cpu().numpy())

y_true = np.concatenate(y_true_list)
y_pred = np.concatenate(y_pred_list)

# ---------- 4️⃣ 计算指标 ----------
test_mse = mean_squared_error(y_true, y_pred)
test_r2  = r2_score(y_true, y_pred)

print("🧪 Test results")
print(f"Test MSE: {test_mse:.4f}")
print(f"Test R² : {test_r2:.4f}")


🧪 Test results
Test MSE: 0.5577
Test R² : 0.4552


/tmp/ipykernel_305195/1227843688.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("/home/charles/HZU/Industrial_Software_Testing/Industr